# Transfer Learning USAD → Detección de Anomalías de Temperatura SIATA

Adapta el modelo USAD pre-entrenado en el dataset SWaT (51 sensores industriales) para detectar
anomalías de temperatura en las 4 estaciones meteorológicas SIATA del Valle de Aburrá, Medellín.

**Estrategia de transfer learning:** Inicialización por submatriz — se copian las submatrices
superiores-izquierdas de los pesos pre-entrenados como punto de partida para la nueva arquitectura
de menor dimensión.

**Referencia:** Audibert et al. (2020). USAD: UnSupervised Anomaly Detection on multivariate time series. KDD 2020.

## Sección 0 — Setup

### Si estás en Google Colab, monta tu Drive primero:
```python
from google.colab import drive
drive.mount('/content/drive')
import os
os.chdir('/content/drive/MyDrive/monografia/modelos/usad')  # ajustar ruta
```

In [3]:
# ─── SETUP GOOGLE COLAB ─────────────────────────────────────────────────────
# Ejecuta esta celda primero si estás en Google Colab.
# Requiere un secret llamado GITHUB_TOKEN:
#   Panel izquierdo > ícono 🔑 Secrets > Add new secret
#   Name: GITHUB_TOKEN  |  Value: tu Personal Access Token de GitHub
import sys, os

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    from google.colab import userdata
    TOKEN    = userdata.get('GITHUB_TOKEN')
    USER     = 'ronvas234'
    REPO     = 'data-science-monograph'
    WORKDIR  = f'/content/{REPO}/modelos/usad'
    REPO_URL = f'https://{TOKEN}@github.com/{USER}/{REPO}.git'

    if not os.path.exists(f'/content/{REPO}/.git'):
        os.system(f'git clone --quiet {REPO_URL} /content/{REPO}')
        print('Repo clonado.')
    else:
        os.system(f'git -C /content/{REPO} pull --quiet')
        print('Repo actualizado.')

    os.chdir(WORKDIR)
    print(f'Directorio: {os.getcwd()}')
    print('Archivos:', os.listdir('.'))


Cloning into '/content/data-science-monograph'...
fatal: could not read Username for 'https://github.com': No such device or address


FileNotFoundError: [Errno 2] No such file or directory: '/content/data-science-monograph/modelos/usad'

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.utils.data as data_utils
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import classification_report

from utils import *
from usad import *

device = get_default_device()
print(f'Device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

## Sección 1 — Análisis Exploratorio de Datos (EDA)

In [ ]:
# Cargar datos
df = pd.read_csv('data_new/temperatura_estaciones_marzo_abril_2025.csv',
                 parse_dates=['fecha_hora'])
print(f'Total filas: {len(df):,}')
print(f'Estaciones: {df["estacion_nombre"].unique()}')
print(f'Periodo: {df["fecha_hora"].min()} → {df["fecha_hora"].max()}')
print(f'\nValores calidad_dudosa=True: {df["calidad_dudosa"].sum():,} ({df["calidad_dudosa"].mean()*100:.2f}%)')
df.head()

In [ ]:
# Temperatura por estación
fig, axes = plt.subplots(4, 1, figsize=(16, 10), sharex=True)
stations = df['estacion_nombre'].unique()
colors = ['#2196F3', '#4CAF50', '#FF9800', '#E91E63']
for ax, station, color in zip(axes, stations, colors):
    sub = df[df['estacion_nombre'] == station]
    ax.plot(sub['fecha_hora'], sub['t'], linewidth=0.3, color=color, alpha=0.8)
    # Marcar puntos anómalos
    anomalies = sub[sub['calidad_dudosa'] == True]
    ax.scatter(anomalies['fecha_hora'], anomalies['t'], color='red', s=2, zorder=5, label='calidad_dudosa')
    ax.set_ylabel('°C')
    ax.set_title(f'{station} | anomalías: {len(anomalies):,}')
    ax.legend(loc='upper right', markerscale=4)
    ax.grid(True, alpha=0.3)
plt.suptitle('Temperatura por Estación SIATA (Marzo-Abril 2025)', y=1.01, fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Estadísticas por estación
stats = df.groupby('estacion_nombre').agg(
    t_min=('t', 'min'),
    t_mean=('t', 'mean'),
    t_max=('t', 'max'),
    t_std=('t', 'std'),
    n_anomalias=('calidad_dudosa', 'sum'),
    pct_anomalias=('calidad_dudosa', 'mean')
)
stats['pct_anomalias'] = (stats['pct_anomalias'] * 100).round(2)
print(stats.to_string())

## Sección 2 — Preprocesamiento de Datos

In [ ]:
# Extraer labels antes de pivotar
labels_long = df[['fecha_hora', 'calidad_dudosa']].copy()

# Pivot: formato largo → ancho (4 columnas de temperatura por estación)
df_wide = df.pivot(index='fecha_hora', columns='codigo', values='t').sort_index()
df_wide.columns = [f'station_{c}' for c in df_wide.columns]
print(f'Forma después del pivot: {df_wide.shape}')  # ~(87577, 4)
print(f'Valores nulos: {df_wide.isnull().sum().to_dict()}')
df_wide.head(3)

In [ ]:
# Label por timestamp: True si CUALQUIER estación tiene calidad dudosa
labels_per_ts = labels_long.groupby('fecha_hora')['calidad_dudosa'].any().astype(int)
labels_per_ts = labels_per_ts.reindex(df_wide.index, fill_value=0)
print(f'Timestamps anómalos: {labels_per_ts.sum():,} / {len(labels_per_ts):,} ({labels_per_ts.mean()*100:.2f}%)')

In [ ]:
# Normalización MinMaxScaler, fit SOLO sobre datos normales (igual que SWaT original)
normal_mask = (labels_per_ts.values == 0)
scaler = MinMaxScaler()
scaler.fit(df_wide.values[normal_mask])
df_scaled = scaler.transform(df_wide.values)
print(f'Rango después de normalizar: [{df_scaled.min():.3f}, {df_scaled.max():.3f}]')

In [ ]:
# Sliding windows
window_size = 60  # 1 hora de contexto a resolución 1-minuto
data = df_scaled
labels_arr = labels_per_ts.values
N = len(data)

# Crear ventanas deslizantes
windows = data[np.arange(window_size)[None, :] + np.arange(N - window_size)[:, None]]
windows = windows.reshape(len(windows), -1)  # (N-ws, window_size*4) = (N-ws, 240)
print(f'Ventanas creadas: {windows.shape}')  # ~(87517, 240)

# Labels por ventana: mayoría simple (>30 de 60 timesteps anómalos)
window_labels = np.array([
    1 if np.sum(labels_arr[i:i + window_size]) > window_size // 2 else 0
    for i in range(N - window_size)
])
print(f'Ventanas anómalas: {window_labels.sum():,} ({window_labels.mean()*100:.2f}%)')
print(f'Ventanas normales: {(1-window_labels).sum():,}')

In [ ]:
# Split entrenamiento/validación: SOLO ventanas 100% limpias (sin ningún timestep anómalo)
clean_mask = np.array([
    np.sum(labels_arr[i:i + window_size]) == 0
    for i in range(N - window_size)
])
clean_windows = windows[clean_mask]
split = int(0.8 * len(clean_windows))
windows_train = clean_windows[:split]
windows_val   = clean_windows[split:]
windows_test  = windows
y_test        = window_labels

print(f'Train: {len(windows_train):,} ventanas limpias')
print(f'Val:   {len(windows_val):,} ventanas limpias')
print(f'Test:  {len(windows_test):,} ventanas (normal + anómalas)')

In [ ]:
# Hiperparámetros del nuevo modelo
w_size     = window_size * 4  # 240
z_size     = 1200             # igual al modelo pre-entrenado → máxima transferencia
BATCH_SIZE = 512

train_loader = data_utils.DataLoader(
    data_utils.TensorDataset(torch.from_numpy(windows_train).float()),
    batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

val_loader = data_utils.DataLoader(
    data_utils.TensorDataset(torch.from_numpy(windows_val).float()),
    batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

test_loader = data_utils.DataLoader(
    data_utils.TensorDataset(torch.from_numpy(windows_test).float()),
    batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

print(f'w_size={w_size}, z_size={z_size}')
print(f'Batches por epoch (train): {len(train_loader)}')

## Sección 3 — Inspección del Modelo Pre-Entrenado y Transfer Learning

In [ ]:
# Inspeccionar dimensiones del modelo pre-entrenado (SWaT)
ckpt = torch.load('model.pth', map_location='cpu')
print('=== Modelo pre-entrenado (SWaT: w_size=612, z_size=1200) ===')
for component in ['encoder', 'decoder1']:
    print(f'\n{component}:')
    for key, val in ckpt[component].items():
        print(f'  {key}: {tuple(val.shape)}')

In [ ]:
# Dimensiones del nuevo modelo (SIATA: w_size=240, z_size=1200)
model_transfer = UsadModel(w_size, z_size)
print('=== Nuevo modelo (SIATA: w_size=240, z_size=1200) ===')
for name, param in model_transfer.named_parameters():
    print(f'  {name}: {tuple(param.shape)}')
total_params = sum(p.numel() for p in model_transfer.parameters())
print(f'\nTotal parámetros: {total_params:,}')

In [ ]:
# Verificar que todas las dimensiones nuevas son <= a las pre-entrenadas
print('Verificación de compatibilidad de dimensiones:')
print(f'{"Capa":<30} {"Pre-entrenado":<20} {"Nuevo":<20} {"Compatible"}')
print('-' * 80)
new_state = dict(model_transfer.named_parameters())
for component in ['encoder', 'decoder1']:
    for key, src in ckpt[component].items():
        full_key = f'{component}.{key}'
        if full_key in new_state:
            dst = new_state[full_key]
            compatible = all(s >= d for s, d in zip(src.shape, dst.shape))
            print(f'{full_key:<30} {str(tuple(src.shape)):<20} {str(tuple(dst.shape)):<20} {"✓" if compatible else "✗"}')

In [ ]:
# Aplicar transfer learning por submatriz
model_transfer = to_device(model_transfer, device)
model_transfer = load_pretrained_submatrix(model_transfer, 'model.pth')

# Verificar: normas de pesos deben ser distintas de una init aleatoria
print('Normas de pesos después de la inicialización por submatriz:')
for name, param in model_transfer.named_parameters():
    print(f'  {name}: norm = {param.data.norm().item():.4f}')

## Sección 4 — Fase 1: Decoder Warm-Up (encoder congelado, 10 épocas)

In [ ]:
# Congelar encoder — el espacio latente pre-entrenado se preserva
for param in model_transfer.encoder.parameters():
    param.requires_grad = False

opt1 = torch.optim.Adam(model_transfer.decoder1.parameters())
opt2 = torch.optim.Adam(model_transfer.decoder2.parameters())

history_phase1 = []
N_EPOCHS_P1 = 10

for epoch in range(N_EPOCHS_P1):
    for [batch] in train_loader:
        batch = to_device(batch, device)
        # Entrenar Decoder1 (AE1)
        loss1, loss2 = model_transfer.training_step(batch, epoch + 1)
        loss1.backward()
        opt1.step()
        opt1.zero_grad()
        # Entrenar Decoder2 (AE2)
        loss1, loss2 = model_transfer.training_step(batch, epoch + 1)
        loss2.backward()
        opt2.step()
        opt2.zero_grad()
    result = evaluate(model_transfer, val_loader, epoch + 1)
    model_transfer.epoch_end(epoch, result)
    history_phase1.append(result)

## Sección 5 — Fase 2: Fine-Tuning Completo (lr=1e-4, 40 épocas)

In [ ]:
# Descongelar encoder
for param in model_transfer.encoder.parameters():
    param.requires_grad = True

N_EPOCHS_P2 = 40
history_phase2 = training(
    N_EPOCHS_P2, model_transfer, train_loader, val_loader,
    opt_func=lambda p: torch.optim.Adam(p, lr=1e-4)
)

# Gráfica combinada de ambas fases
plot_history(history_phase1 + history_phase2)

## Sección 6 — Guardar Modelo

In [ ]:
torch.save({
    'encoder':  model_transfer.encoder.state_dict(),
    'decoder1': model_transfer.decoder1.state_dict(),
    'decoder2': model_transfer.decoder2.state_dict()
}, 'model_siata_transfer.pth')
print('Modelo guardado en model_siata_transfer.pth')

## Sección 7 — Evaluación

In [ ]:
# Calcular scores de anomalía para todo el test set
results = testing(model_transfer, test_loader, alpha=0.5, beta=0.5)
y_pred = np.concatenate([
    torch.stack(results[:-1]).flatten().detach().cpu().numpy(),
    results[-1].flatten().detach().cpu().numpy()
])
print(f'Scores calculados: {len(y_pred):,}')
print(f'Score range: [{y_pred.min():.6f}, {y_pred.max():.6f}]')

In [ ]:
# Curva ROC y threshold óptimo
threshold = ROC(y_test, y_pred)

In [ ]:
# Histograma de scores: normal vs anómalo
histogram(y_test, y_pred)

In [ ]:
# Matriz de confusión al threshold óptimo
y_pred_binary = (y_pred >= threshold).astype(int)
confusion_matrix(y_test, y_pred_binary)
print('\nReporte de clasificación:')
print(classification_report(y_test, y_pred_binary, target_names=['Normal', 'Anómalo']))

In [ ]:
# Visualizar scores sobre la serie de tiempo de temperatura
timestamps = df_wide.index[window_size:]  # timestamps correspondientes a cada ventana
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(16, 8), sharex=True)

# Temperatura de una estación
ax1.plot(timestamps, df_scaled[window_size:, 0], linewidth=0.5, color='#2196F3', label='station_68 (normalizada)')
ax1.set_ylabel('Temperatura (normalizada)')
ax1.set_title('Temperatura Jardín Botánico vs Score de Anomalía')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Score de anomalía
ax2.plot(timestamps, y_pred, linewidth=0.5, color='#FF5722', label='Score anomalía')
ax2.axhline(y=threshold, color='red', linestyle='--', linewidth=1.5, label=f'Threshold = {float(threshold):.6f}')
ax2.fill_between(timestamps, y_pred, threshold,
                 where=(y_pred >= threshold), alpha=0.3, color='red', label='Detectado como anómalo')
ax2.set_ylabel('Score USAD')
ax2.set_xlabel('Fecha')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Sección 8 — Ablación: Comparación con Baselines

Para la monografía: comparar el modelo con transfer learning contra dos baselines entrenados desde cero
con la misma arquitectura y datos.

In [ ]:
# Baseline A: misma arquitectura (w=240, z=1200), inicialización Xavier aleatoria, 50 épocas
print('=== Baseline A: Xavier aleatorio, z_size=1200, 50 épocas ===')
model_baseline_a = UsadModel(w_size, z_size)
model_baseline_a = to_device(model_baseline_a, device)
history_ba = training(50, model_baseline_a, train_loader, val_loader)

In [ ]:
# Baseline B: arquitectura reducida (w=240, z=100), inicialización Xavier aleatoria, 50 épocas
print('=== Baseline B: Xavier aleatorio, z_size=100, 50 épocas ===')
model_baseline_b = UsadModel(w_size, 100)
model_baseline_b = to_device(model_baseline_b, device)
history_bb = training(50, model_baseline_b, train_loader, val_loader)

In [ ]:
from sklearn.metrics import roc_auc_score

def get_scores(model, loader):
    results = testing(model, loader, alpha=0.5, beta=0.5)
    return np.concatenate([
        torch.stack(results[:-1]).flatten().detach().cpu().numpy(),
        results[-1].flatten().detach().cpu().numpy()
    ])

y_pred_ba = get_scores(model_baseline_a, test_loader)
y_pred_bb = get_scores(model_baseline_b, test_loader)

auc_transfer = roc_auc_score(y_test, y_pred)
auc_ba       = roc_auc_score(y_test, y_pred_ba)
auc_bb       = roc_auc_score(y_test, y_pred_bb)

print('=== Comparación AUC-ROC ===')
print(f'Transfer Learning (submatriz, z=1200): AUC = {auc_transfer:.4f}')
print(f'Baseline A (aleatorio, z=1200):        AUC = {auc_ba:.4f}')
print(f'Baseline B (aleatorio, z=100):         AUC = {auc_bb:.4f}')

In [ ]:
# Comparar curvas de convergencia
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

h_transfer = history_phase1 + history_phase2
for h, label, color in [
    (h_transfer, 'Transfer (submatriz)', '#2196F3'),
    (history_ba, 'Baseline A (z=1200)', '#4CAF50'),
    (history_bb, 'Baseline B (z=100)',  '#FF9800')
]:
    losses1 = [x['val_loss1'] for x in h]
    losses2 = [x['val_loss2'] for x in h]
    ax1.plot(losses1, '-', label=label, color=color)
    ax2.plot(losses2, '-', label=label, color=color)

ax1.set_title('val_loss1 (AE1 — reconstrucción)')
ax1.set_xlabel('Época'); ax1.set_ylabel('Loss')
ax1.legend(); ax1.grid(True, alpha=0.3)

ax2.set_title('val_loss2 (AE2 — adversarial)')
ax2.set_xlabel('Época'); ax2.set_ylabel('Loss')
ax2.legend(); ax2.grid(True, alpha=0.3)

plt.suptitle('Comparación de Convergencia: Transfer vs Baselines', fontsize=13)
plt.tight_layout()
plt.show()

## Sección 9 — Prueba Rápida (Sanity Check)

Verifica que el modelo guardado carga correctamente y produce scores esperados.

In [ ]:
# Cargar modelo guardado y verificar reproducibilidad
checkpoint = torch.load('model_siata_transfer.pth', map_location=device)
model_loaded = UsadModel(w_size, z_size)
model_loaded = to_device(model_loaded, device)
model_loaded.encoder.load_state_dict(checkpoint['encoder'])
model_loaded.decoder1.load_state_dict(checkpoint['decoder1'])
model_loaded.decoder2.load_state_dict(checkpoint['decoder2'])

# Prueba con ventana anómala vs normal conocida
anomaly_indices = np.where(y_test == 1)[0]
normal_indices  = np.where(y_test == 0)[0]

if len(anomaly_indices) > 0:
    idx_a = anomaly_indices[0]
    idx_n = normal_indices[0]

    s_anomaly = to_device(torch.from_numpy(windows_test[idx_a:idx_a+1]).float(), device)
    s_normal  = to_device(torch.from_numpy(windows_test[idx_n:idx_n+1]).float(), device)

    with torch.no_grad():
        def score_single(model, x):
            w1 = model.decoder1(model.encoder(x))
            w2 = model.decoder2(model.encoder(w1))
            return (0.5 * torch.mean((x - w1)**2) + 0.5 * torch.mean((x - w2)**2)).item()

        sc_a = score_single(model_loaded, s_anomaly)
        sc_n = score_single(model_loaded, s_normal)

    print(f'Score ventana ANÓMALA:  {sc_a:.8f}')
    print(f'Score ventana NORMAL:   {sc_n:.8f}')
    print(f'Anomalía > Normal:      {sc_a > sc_n} ← debe ser True')
else:
    print('No hay ventanas anómalas en el test set (revisar labeling)')